In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))
from utils.load_utils import load_parameter_results

In [2]:
_PROJ     = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita"
RESULTS_DIR = f"{_PROJ}/optimizations/flames"

FOLDS   = [0, 1, 2, 3]
LAMBDAS = [0.01, 0.1, 1.0, 10.0]

In [3]:
# Lambda sweep
df_lambda = load_parameter_results(RESULTS_DIR, "lambda", LAMBDAS, folds=FOLDS)

Total windows loaded: 696


In [4]:
df_lambda["no_edits"] = df_lambda["n_edits"] == 0

no_edits_summary = (
    df_lambda.groupby("lambda")["no_edits"]
    .agg(n_total="count", n_no_edits="sum")
    .assign(pct_no_edits=lambda x: 100 * x["n_no_edits"] / x["n_total"])
)
print(no_edits_summary.to_string())

        n_total  n_no_edits  pct_no_edits
lambda                                   
0.01        174           0           0.0
0.10        174           0           0.0
1.00        174           0           0.0
10.00       174           0           0.0


In [7]:
df_lambda.columns

Index(['chrom', 'fold', 'PearsonR', 'centered_start', 'centered_end',
       'centered_flat_start', 'centered_flat_end', 'active_fraction',
       'neutral_fraction', 'repressive_fraction', 'n_edits',
       'last_accepted_step', 'flame3_orig', 'flame3_edited', 'flame3_target',
       'flame5_orig', 'flame5_edited', 'flame5_target', 'flame7_orig',
       'flame7_edited', 'flame7_target', 'GC_seq', 'GC_slice_orig',
       'GC_slice_edited', 'init_CTCFs_num', 'CTCFs_num', 'FIMO_sum',
       'FIMO_max', 'orientation', 'positions', 'lambda', 'no_edits'],
      dtype='object')

In [8]:
df_lambda["flame_diff"] = df_lambda["flame3_edited"] - df_lambda["flame3_orig"]
df_lambda["no_improvement"] = df_lambda["flame_diff"] <= 0

no_improvement_summary = (
    df_lambda.groupby("lambda")["no_improvement"]
    .agg(n_total="count", n_no_improvement="sum")
    .assign(pct_no_improvement=lambda x: 100 * x["n_no_improvement"] / x["n_total"])
)
print(no_improvement_summary.to_string())

        n_total  n_no_improvement  pct_no_improvement
lambda                                               
0.01        174                 0                 0.0
0.10        174                 0                 0.0
1.00        174                 0                 0.0
10.00       174                 0                 0.0


In [9]:
success_rate = (
    df_lambda.assign(success=~df_lambda["no_edits"] & ~df_lambda["no_improvement"])
    .groupby("lambda")["success"]
    .agg(n_total="count", n_success="sum")
    .assign(pct_success=lambda x: 100 * x["n_success"] / x["n_total"])
)
print(success_rate.to_string())

        n_total  n_success  pct_success
lambda                                 
0.01        174        174        100.0
0.10        174        174        100.0
1.00        174        174        100.0
10.00       174        174        100.0


In [11]:
summary = (
    df_lambda.groupby("lambda")
    .agg(
        n_success        = ("n_edits",          "count"),
        mean_n_edits     = ("n_edits",          "mean"),
        mean_flame_score_diff  = ("flame_diff",        "mean"),
        mean_CTCFs       = ("CTCFs_num",         "mean"),
    )
    .round(3)
)
print(summary.to_string())

        n_success  mean_n_edits  mean_flame_score_diff  mean_CTCFs
lambda                                                            
0.01          174       744.730                  0.715       4.310
0.10          174       737.747                  0.725       4.678
1.00          174       672.063                  0.709       4.425
10.00         174       376.879                  0.704       4.460
